# HAGI — GDR Ablation (one push, run top-to-bottom)

Trains the four ablation models (**A/B/C/D**), each to its **own private HF repo**,
resuming from Drive **and** HF so a killed session never loses progress.

Order is **B, D, C, A** — the decisive pair (B vs D) lands first, so if A100 units
run out mid-way you still have the headline result. Each model is independent.

**Before running:** Runtime → Change runtime type → **A100 GPU** → Save. The
`HF_TOKEN` (write) secret and the Stage-0 shards on Drive must already exist.

## 1. Clone + update the repo (experimental branch)

In [ ]:
import os
%cd /content
if not os.path.isdir('HAGI'):
    !git clone -b experimental https://github.com/ShmidtS/HAGI.git
%cd /content/HAGI
!git pull --ff-only origin experimental   # ensure latest configs + --train-tokens flag + generate.py
print('cwd:', os.getcwd())

## 2. Install dependencies (~3-5 min the first time)

In [ ]:
!pip install -q -r requirements.txt

## 3. Drive + HF token + knobs

Edit `HF_USER` if needed. Lower `TRAIN_TOKENS` (e.g. `300_000_000`) or trim `ORDER`
if the A100 budget is tight — the LR schedule adapts to the token count.

In [ ]:
import os, torch
from google.colab import drive, userdata

drive.mount('/content/drive')
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

# ---- knobs ---------------------------------------------------------------
HF_USER      = 'NAME0x0'                            # your HF username
DATA_DIR     = '/content/drive/MyDrive/hagi-data'   # existing Stage-0 shards
CKPT_ROOT    = '/content/drive/MyDrive/hagi-ckpts'  # checkpoints persist on Drive
TRAIN_TOKENS = 500_000_000                          # per model; lower if budget tight
ORDER        = ['b', 'd', 'c', 'a']                 # decisive pair first; drop tail if needed
# --------------------------------------------------------------------------

CONFIGS = {m: f'configs/ablation_{m}.yaml' for m in ['a', 'b', 'c', 'd']}
REPOS   = {m: f'{HF_USER}/hagi-ablation-{m}' for m in ['a', 'b', 'c', 'd']}
TOKENS_PER_STEP = 16 * 4 * 1024                     # batch * accum * seq = 65,536
STEPS   = TRAIN_TOKENS // TOKENS_PER_STEP + 200     # > max_steps -> finishes in one session

print('HF token loaded.')
print(f'plan: {ORDER}  |  {TRAIN_TOKENS:,} tok/model  |  ~{STEPS} steps/model')
for m in ORDER:
    print(f'  {m}: {CONFIGS[m]:<24} -> {REPOS[m]}')

## 4. Preflight (fails fast, before any A100 time is spent)
Checks GPU class, shards on Drive, HF token validity, and creates the four repos.

In [ ]:
import glob
from huggingface_hub import HfApi

assert torch.cuda.is_available(), 'No GPU. Runtime -> Change runtime type -> A100.'
p  = torch.cuda.get_device_properties(0)
sm = p.major * 10 + p.minor
print(f'GPU: {p.name} | sm{sm} | {p.total_memory/1e9:.0f} GB')
if not (sm >= 80 and p.total_memory/1e9 >= 22):
    print('WARNING: not an A100-class GPU; configs are tuned for A100-40GB (may OOM/slow).')

shards = glob.glob(f'{DATA_DIR}/*.bin')
assert shards, f'no .bin shards in {DATA_DIR} - check DATA_DIR / Drive mount'
print(f'shards: {len(shards)} .bin files in {DATA_DIR}')

api = HfApi()
print('HF user:', api.whoami()['name'])
for m in ORDER:
    api.create_repo(REPOS[m], repo_type='model', private=True, exist_ok=True)
    print('  repo ready:', REPOS[m])

units = {'a': 11, 'b': 19, 'c': 12, 'd': 19}
scale = TRAIN_TOKENS / 500_000_000
print('\nest. A100 units (@%.0fM): ' % (TRAIN_TOKENS/1e6) +
      ', '.join(f'{m}~{units[m]*scale:.0f}' for m in ORDER) +
      f'  | total ~{sum(units[m] for m in ORDER)*scale:.0f}')

## 5. Smoke test (cheap insurance, ~2-3 min)
Runs the **superset** model D (loop + GDR) for 15 steps to `/tmp` with **no HF push**.
If D's path works, A/B/C (feature subsets) work too. Catches config/data/compile/OOM
errors **before** spending real units.

> If this OOMs: set `gradient_checkpointing: true` in `configs/ablation_d.yaml` and
> `configs/ablation_b.yaml`, then re-run from cell 1.

In [ ]:
import glob, subprocess
smoke_dir = '/tmp/smoke'
r = subprocess.run(
    ['python', '-u', '-m', 'prototype.training.train',   # -u: unbuffered -> logs stream live
     '--config', CONFIGS['d'], '--data', DATA_DIR, '--device', 'cuda',
     '--ckpt-dir', smoke_dir, '--steps', '15', '--train-tokens', '2000000'],
    cwd='/content/HAGI')
assert r.returncode == 0, 'smoke run failed - read the traceback above'
ck = glob.glob(f'{smoke_dir}/ablation_d/step-*.pt')
assert ck, 'smoke produced no checkpoint'
print('\nSMOKE OK ->', ck[0])
print('pipeline verified (build, Drive data, compile, train, checkpoint). Safe to run full.')

## 6. Train all selected models (resumable, one HF repo each)
Each model runs to completion in one session. A failure is recorded and the others
still run; **re-run this cell** and `--resume auto` continues any unfinished model
from its Drive/HF checkpoint. Safe to interrupt and re-run anytime.

In [ ]:
import subprocess, time
results = {}
for m in ORDER:
    cfg, repo = CONFIGS[m], REPOS[m]
    print(f'\n{"="*70}\nMODEL {m.upper()}   {cfg}  ->  {repo}\n{"="*70}', flush=True)
    t0 = time.time()
    r = subprocess.run(
        ['python', '-u', '-m', 'prototype.training.train',   # -u: unbuffered -> logs stream live
         '--config', cfg, '--data', DATA_DIR, '--device', 'cuda',
         '--ckpt-dir', CKPT_ROOT, '--hf-repo', repo,
         '--resume', 'auto', '--steps', str(STEPS),
         '--train-tokens', str(TRAIN_TOKENS)],
        cwd='/content/HAGI')
    dt = (time.time() - t0) / 60
    results[m] = (r.returncode == 0)
    print(f'[{m}] {"DONE" if results[m] else "FAILED"} in {dt:.0f} min')
    if not results[m]:
        print(f'[{m}] failed - others continue; re-run this cell to resume {m}.')

print('\nSUMMARY:', {m: ('ok' if results[m] else 'FAILED') for m in ORDER})

## 7. Sample check (sanity — generates from each local checkpoint)

In [ ]:
import glob, subprocess
for m in ORDER:
    cks = sorted(glob.glob(f'{CKPT_ROOT}/ablation_{m}/step-*.pt'))
    if not cks:
        print(f'[{m}] no local checkpoint to sample'); continue
    print(f'\n--- model {m.upper()}  ({cks[-1].split("/")[-1]}) ---')
    subprocess.run(
        ['python', '-u', 'scripts/generate.py', '--ckpt', cks[-1],
         '--prompt', 'The sun is a star that', '--max-new-tokens', '40', '--device', 'cuda'],
        cwd='/content/HAGI')

## 8. Summary

In [ ]:
print('HF repos (private):')
for m in ORDER:
    print(f'  {m}: https://huggingface.co/{REPOS[m]}')
print('\nNext: compare final train loss / perplexity per model.')
print('Decisive: B vs D (grade decomposition isolated). Secondary: C vs D. See docs/ABLATION.md.')

## 9. Compare — the result

Scores every model on one shared fixed batch set (identical data), prints loss +
perplexity, and reports **D-B** (negative = grade decomposition helps) and **D-C**.
This is the experiment's answer. Cheap — no GPU training, ~1 min.

In [ ]:
# Score all trained models on the SAME fixed batches -> loss + perplexity.
# Decisive: B vs D. Cheap (no training) — runs on the A100 in ~1 min.
import glob, subprocess
ckpts = []
for m in ORDER:
    cks = sorted(glob.glob(f'{CKPT_ROOT}/ablation_{m}/step-*.pt'))
    if cks:
        ckpts.append(cks[-1])
subprocess.run(
    ['python', '-u', 'scripts/eval_loss.py',
     '--data', DATA_DIR, '--device', 'cuda', '--batches', '50', '--ckpt', *ckpts],
    cwd='/content/HAGI')